In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load data (handle large file)
data_path = Path('../data/raw/complaints.csv')
df = pd.read_csv(data_path, low_memory=False)
print(df.shape)  # Expected: ~ (millions of rows, 18 columns)
print(df.head())

# EDA: Distribution across Products
product_counts = df['Product'].value_counts()
plt.figure(figsize=(12, 6))
sns.barplot(x=product_counts.index, y=product_counts.values)
plt.xticks(rotation=90)
plt.title('Complaint Distribution by Product')
plt.savefig('../notebooks/product_distribution.png')  # Save for report
plt.show()

# Word count of narratives
df['narrative_length'] = df['Consumer complaint narrative'].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)
plt.figure(figsize=(10, 5))
sns.histplot(df['narrative_length'], bins=50, kde=True)
plt.title('Distribution of Narrative Word Counts')
plt.xlabel('Word Count')
plt.savefig('../notebooks/narrative_lengths.png')
plt.show()
print(f"Median word count: {df['narrative_length'].median()}")
print(f"Complaints with narratives: {df['Consumer complaint narrative'].notnull().sum()}")
print(f"Complaints without narratives: {df['Consumer complaint narrative'].isnull().sum()}")

# Identify very short/long: e.g., <10 words or >1000
short = df[df['narrative_length'] < 10].shape[0]
long = df[df['narrative_length'] > 1000].shape[0]
print(f"Very short narratives: {short}, Very long: {long}")

ModuleNotFoundError: No module named 'pandas'

### Filter and Clean

In [ ]:
# Filter to specified products (adjust strings based on unique values in df['Product'].unique())
products = ['Credit card', 'Consumer Loan', 'Checking or savings account', 'Money transfer, money service, or virtual currency']
df_filtered = df[df['Product'].isin(products)].copy()

# Remove empty narratives
df_filtered = df_filtered[df_filtered['Consumer complaint narrative'].notnull()]

# Clean narratives
def clean_text(text):
    text = text.lower()  # Lowercase
    text = ''.join(c for c in text if c.isalnum() or c.isspace())  # Remove special chars
    text = text.replace('i am writing to file a complaint', '')  # Remove boilerplate (example)
    return text.strip()

df_filtered['clean_narrative'] = df_filtered['Consumer complaint narrative'].apply(clean_text)

# Save
filtered_path = Path('../data/processed/filtered_complaints.csv')
df_filtered.to_csv(filtered_path, index=False)
print(f"Filtered shape: {df_filtered.shape}")